# TV series popularity: exploration notes

**Task:** find what is associated with popularity for Netflix TV series, then outline an ML approach to predict popularity before launch.

**Cleaning Data:** Within this notebook I will be cleaning and understanding the data before I do start a full EDA. `raw` is the unmodified data. Nothing is cleaned until the exploration says what needs cleaning. Every cleaning step gets a row in the cleaning log at the bottom.

In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [2]:
from tmdb_tv_shows.clean import load_raw, build

raw = load_raw()   # the file as read, never changed


In [3]:
# print(raw.shape)
# raw.info()

In [4]:
raw.describe().T

,count,mean,std,min,25%,50%,75%,max
id,168639.0,111307.074704,76451.662352,1.0,45936.5,97734.000,196923.5000,251213.000
number_of_seasons,168639.0,1.548497,2.942872,0.0,1.0,1.000,1.0000,240.000
number_of_episodes,168639.0,24.465082,134.799622,0.0,1.0,6.000,20.0000,20839.000
vote_count,168639.0,13.305054,190.809059,0.0,0.0,0.000,1.0000,21857.000
vote_average,168639.0,2.333843,3.454334,0.0,0.0,0.000,6.0000,10.000
popularity,168639.0,5.882644,42.023216,0.0,0.6,0.857,2.4315,3707.008
episode_run_time,168639.0,22.603348,47.950427,0.0,0.0,0.000,42.0000,6032.000


**Observations**
- `name` count > uniqiue, --> May need investigating as they could be duplicates
- `ID` repeated ids
- `number_of_seasons` and `number_of_episodes` have outlier at max --> Determine if they are valid or not
- `vote_count`,`vote_average` and `popularity` --> maybe considered as key indicators
- `vote_counts` are skewed
- `overview` repeats --> duplicates?
- `genre` needs to be seperated
- `created_by` may give insight
- Placeholders within columns such as `episode_run_time`,`number_of_episodes` and `popularity`
- `votes` can be considered as unreliable
- convert columsn to there respective format

## Cleaning

All cleaning code is in `src/tmdb_tv_shows/clean.py`, so it can be reused. The decisions behind each step are in the notes below and in the cleaning log at the bottom.

- `clean`: all cleaning applied, repeated IDs still present
- `dedup`: one row per show, use this for analysis
- `dropped`: the copies not kept, nothing is lost

In [5]:
raw, clean, dedup, dropped = build(raw)

# fail loudly if the cleaning code changes
assert len(clean) == 167_059
assert len(dedup) == 164_705 and dedup.id.is_unique
assert len(dropped) == 2_354
assert dedup.is_netflix.sum() == 1_929
print(len(raw), len(clean), len(dedup), len(dropped))   # 168639 167059 164705 2354

168639 167059 164705 2354


## Netflix subset

In [6]:
netflix = dedup[dedup.is_netflix].copy()

print(len(netflix), netflix.id.nunique())   # expect 1929 1929
netflix.describe(include="all").T

1929 1929


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
id,1929.0,NaN,NaN,NaN,135048.303266,1424.0,88393.0,114363.0,203704.0,250850.0,58512.450462
name,1929,1926,The Chosen One,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
number_of_seasons,1929.0,NaN,NaN,NaN,1.572317,1.0,1.0,1.0,2.0,30.0,1.51585
number_of_episodes,1928.0,NaN,NaN,NaN,14.865145,0.0,6.0,8.0,16.0,973.0,33.804737
original_language,1929,33,en,1079,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vote_count,1929.0,NaN,NaN,NaN,194.827372,0.0,5.0,21.0,86.0,17836.0,897.353206
vote_average,1708.0,NaN,NaN,NaN,6.867374,1.5,6.4,7.1,7.6005,10.0,1.264277
overview,1872,1871,"Away from school, during the winter holidays, ...",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
adult,1929,2,False,1927,NaN,NaN,NaN,NaN,NaN,NaN,NaN
backdrop_path,1794,1792,/jsBCGPk1z4WJvmGRKfmXono3PtL.jpg,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### ID repeats

- `id` repeats often: 1,371 IDs have more than one copy (2,354 extra rows). In the Netflix subset, 28 extra rows across 16 shows.
- `popularity` differs most. 906 IDs differ only in popularity. 330 differ in `last_air_date`.
- **Assumed:** the copies are snapshots taken at different times. There is no capture timestamp.

**Which copy is kept, in order**

1. a copy that lists Netflix (3 Netflix IDs have a blank-network copy)
2. if `first_air_date` is the same across copies: latest `last_air_date`, then most filled columns. If it differs: most filled columns, then latest `last_air_date`
3. where only `popularity` differs: popularity becomes the median of the copies (836 values changed), original kept in `popularity_raw`

**Repeated measurements:** within-show spread of log popularity is 0.38, against 0.66 between repeated shows. One popularity reading is noisy. Not extra training rows, copies are not independent.

## Decided to leave

- **`popularity` 0:** 2,194 rows. Smallest positive value is 0.3, so 0 looks like a floor. Kept as 0.
- **Pilot episodes:** 20 rows missing. Not imputed. If used, impute at modelling time on training data only, with a flag.
- **Outliers:** seasons and episodes are real long-running shows. `episode_run_time` has 99 rows above 600 minutes, kept as is.
- **Future start dates:** 32 rows after June 2024, none Netflix. Handle in EDA by limiting to launched shows.
- **Shared names:** The Lørenskog Disappearance is one show under two IDs. Not removed, a fix for one ID does not scale.
- **Low-vote noise:** `vote_average` is unreliable for shows with few votes. Handle in EDA or modelling.

## Cleaning log

| Step | Why | Effect |
|---|---|---|
| Blank text to missing | blank and NaN should mean the same | `name` (1), `overview` (16), `homepage` (2) |
| Drop exact duplicate rows | identical in every column | 168,639 to 167,059 rows |
| `episode_run_time` 0 to missing | a show cannot run 0 minutes | 81,828 missing after one row per ID |
| Seasons and episodes 0 to missing, except Planned and In Production | 0 is believable only for unreleased shows | 22,401 and 23,868 missing |
| `vote_average` to missing where 0 votes and average 0 | a 0 average needs a vote | 107,327 rows |
| `votes_inconsistent` flag | 0 votes but an average above 0 | 92 rows, values untouched |
| `name_clean` | match names differing by case or spacing | 155,585 to 154,912 distinct names |
| Split list columns into `_list` | one value per item, suffixes re-joined | 6 columns |
| Dates to datetime | text to real dates | no parse failures |
| `dates_reversed` flag | last air date before first air date | 65 rows, values untouched |
| One row per ID | repeated IDs | 167,059 to 164,705 rows, 2,354 kept in `dropped` |